# 04 · Lay → clinical alignment (SapBERT) and the euphemism gap
Measures how far each register's surface forms sit from their clinical concept **before** and **after** contrastive alignment. Train/val split is by phrase.

In [ ]:
# Colab setup: clone the repo and install training extras
import os
import subprocess
import sys

if "google.colab" in sys.modules and not os.path.exists("RareCare"):
    subprocess.run(["git", "clone", "https://github.com/nush1729/RareCare.git"], check=True)
    os.chdir("RareCare")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train]"], check=True)
print("cwd:", os.getcwd())

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")
SEED = 13

In [ ]:
import json

import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

from rarecare.kg.graph import default_graph

kg = default_graph()
surface = {k: v for k, v in json.load(open("training/data/surface_forms.json")).items() if not k.startswith("_")}

def gap_report(model_id):
    tok, mdl = AutoTokenizer.from_pretrained(model_id), AutoModel.from_pretrained(model_id).to(device).eval()
    def emb(t):
        with torch.no_grad():
            e = tok(t, padding=True, truncation=True, max_length=32, return_tensors="pt").to(device)
            return F.normalize(mdl(**e).last_hidden_state[:, 0], dim=-1)
    labels = sorted(surface); L = emb([kg.concepts[c].label for c in labels])
    out = {}
    for reg in ["clinical", "lay", "euphemistic", "hinglish"]:
        hits, n = 0, 0
        for i, c in enumerate(labels):
            for ph in surface[c].get(reg, []):
                hits += int((emb([ph]) @ L.T).argmax() == i); n += 1
        out[reg] = hits / max(1, n)
    return out

before = {m: gap_report(m) for m in ["dmis-lab/biobert-base-cased-v1.2", "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"]}
before

In [ ]:
!python -m training.train_alignment --out checkpoints/sapbert-aligned --epochs 10

In [ ]:
after = gap_report("checkpoints/sapbert-aligned")
import pandas as pd

pd.DataFrame({**before, "sapbert-aligned": after}).T.style.format("{:.2f}")

> ⚠️ This notebook's in-vocabulary check overlaps with training phrases. The reportable euphemism-gap number is computed on the **human-written test set** in notebook 07.